# 🧼 SYSTÈME DE COMPTAGE AUTOMATIQUE DE SAVONS
## Huileries Belhassane - Détection + Tracking + Comptage + Anomalies

**Étapes:**
1. Auto-annotation (si besoin)
2. Entraînement YOLO
3. Processing vidéo complet

---

## 📦 CELL 1: Installation dépendances

In [ ]:
# Installation une seule fois
!pip install ultralytics opencv-python pandas scikit-image pyyaml -q
print("✅ Dépendances installées!")

## 🔧 CELL 2: Imports

In [ ]:
import cv2
import numpy as np
import os
import yaml
import pandas as pd
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("✅ Tous les imports OK!")

## 📁 CELL 3: Configuration chemins

In [ ]:
# À ADAPTER SELON TON CHEMIN!
DATASET_ROOT = "dataset"  # Dossier avec images/ et labels/
VIDEO_PATH = "video.mp4"   # Ta vidéo
OUTPUT_DIR = "results"
MODEL_PATH = "runs/detect/savon_detector/weights/best.pt"
COUNTING_LINE_X = 0.85  # 85% à droite (sortie)

# Créer dossiers
Path(OUTPUT_DIR).mkdir(exist_ok=True)
Path(f"{DATASET_ROOT}/labels").mkdir(parents=True, exist_ok=True)

print(f"✅ Config:")
print(f"   Dataset: {DATASET_ROOT}")
print(f"   Vidéo: {VIDEO_PATH}")
print(f"   Output: {OUTPUT_DIR}")

---
# 🔍 ÉTAPE 1: AUTO-ANNOTATION (si besoin)
## Exécuter si tes 70 images ne sont PAS encore annotées

## CELL 4: Fonction auto-annotation par couleur

In [ ]:
def detect_savons_color(img_path):
    """Détecte savons par plage couleur HSV"""
    img = cv2.imread(img_path)
    if img is None:
        return []
    
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Range pour savons marron/beige (AJUSTER si besoin!)
    lower_brown = np.array([10, 60, 50])
    upper_brown = np.array([25, 255, 200])
    
    mask = cv2.inRange(hsv, lower_brown, upper_brown)
    
    # Morphologie
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    boxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 300 < area < 50000:
            x, y, bw, bh = cv2.boundingRect(cnt)
            if bw > 0 and bh > 0:
                ratio = max(bw, bh) / min(bw, bh)
                if ratio < 5:
                    boxes.append((x, y, x + bw, y + bh))
    
    return boxes

def boxes_to_yolo(boxes, img_width, img_height):
    """Convertit boxes en format YOLO"""
    yolo_lines = []
    for x1, y1, x2, y2 in boxes:
        cx = ((x1 + x2) / 2) / img_width
        cy = ((y1 + y2) / 2) / img_height
        bw = (x2 - x1) / img_width
        bh = (y2 - y1) / img_height
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return yolo_lines

print("✅ Fonctions auto-annotation OK!")

## CELL 5: Exécuter auto-annotation

In [ ]:
# AUTO-ANNOTATION
images_dir = os.path.join(DATASET_ROOT, "images")
labels_dir = os.path.join(DATASET_ROOT, "labels")

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
print(f"🔍 {len(image_files)} images trouvées\n")

total_boxes = 0
for img_file in image_files:
    img_path = os.path.join(images_dir, img_file)
    
    # Détection
    boxes = detect_savons_color(img_path)
    
    # Conversion YOLO
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    yolo_lines = boxes_to_yolo(boxes, w, h)
    
    # Écrire fichier .txt
    txt_filename = img_file.rsplit('.', 1)[0] + '.txt'
    txt_path = os.path.join(labels_dir, txt_filename)
    
    with open(txt_path, 'w') as f:
        for line in yolo_lines:
            f.write(line + '\n')
    
    total_boxes += len(boxes)
    status = "✓" if len(boxes) > 0 else "⚠️"
    if len(image_files) <= 10 or len(image_files) % 7 == 0:
        print(f"{status} {img_file}: {len(boxes)} box(es)")

print(f"\n✅ Auto-annotation terminée!")
print(f"   Total boxes: {total_boxes}")
print(f"   Moyenne par image: {total_boxes/len(image_files):.1f}")

---
# 🤖 ÉTAPE 2: ENTRAÎNEMENT YOLO

## CELL 6: Créer data.yaml

In [ ]:
# Créer data.yaml pour YOLO
data_yaml = {
    'path': os.path.abspath(DATASET_ROOT),
    'train': 'images',
    'val': 'images',
    'nc': 1,  # 1 classe: savon
    'names': ['savon']
}

yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ data.yaml créé: {yaml_path}")
print(f"   Path: {data_yaml['path']}")
print(f"   Classes: {data_yaml['names']}")

## CELL 7: Entraîner YOLO (⏱️ 10-15 min)

In [ ]:
print("🚀 Entraînement YOLO en cours...")
print("   Cela peut prendre 10-15 min selon votre PC\n")

# Charger modèle YOLOv8s
model = YOLO('yolov8s.pt')

# Entraîner
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    device=0,  # GPU (ou 'cpu' si pas GPU)
    save=True,
    project='runs/detect',
    name='savon_detector',
    verbose=False
)

print(f"\n✅ Entraînement terminé!")
print(f"✅ Meilleur modèle: {MODEL_PATH}")

---
# 🎯 ÉTAPE 3: PROCESSING VIDÉO COMPLET
## Détection + Tracking + Comptage + Anomalies

## CELL 8: Classe Tracker simple

In [ ]:
class SimpleTracker:
    def __init__(self, max_age=30):
        self.tracks = {}
        self.next_id = 0
        self.max_age = max_age
        self.frame_count = 0
        
    def update(self, detections):
        self.frame_count += 1
        
        # Centroïdes des détections
        centroids = []
        for det in detections:
            x1, y1, x2, y2 = det[:4]
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            centroids.append((cx, cy, det))
        
        # Matching
        matched_ids = {}
        for track_id, track_data in self.tracks.items():
            if track_data['age'] > self.max_age:
                continue
            
            best_dist = float('inf')
            best_idx = -1
            
            for idx, (cx, cy, det) in enumerate(centroids):
                if idx in matched_ids:
                    continue
                
                dist = np.sqrt((cx - track_data['cx'])**2 + (cy - track_data['cy'])**2)
                if dist < best_dist and dist < 50:
                    best_dist = dist
                    best_idx = idx
            
            if best_idx >= 0:
                matched_ids[best_idx] = track_id
                cx, cy, det = centroids[best_idx]
                self.tracks[track_id]['cx'] = cx
                self.tracks[track_id]['cy'] = cy
                self.tracks[track_id]['det'] = det
                self.tracks[track_id]['age'] = 0
        
        # Results
        results = []
        for track_id, track_data in self.tracks.items():
            if track_data['age'] <= self.max_age:
                det = track_data['det']
                results.append((*det[:4], track_id))
        
        # Nouvelles tracks
        for idx, (cx, cy, det) in enumerate(centroids):
            if idx not in matched_ids:
                self.tracks[self.next_id] = {'cx': cx, 'cy': cy, 'det': det, 'age': 0}
                results.append((*det[:4], self.next_id))
                self.next_id += 1
        
        # Age
        for track_id in self.tracks:
            self.tracks[track_id]['age'] += 1
        
        return results

print("✅ Classe Tracker OK!")

## CELL 9: Classe Anomaly Detector

In [ ]:
class AnomalyDetector:
    def __init__(self):
        self.anomalies = defaultdict(list)
    
    def check_packaging_damaged(self, crop, threshold=0.3):
        """Packaging déchiré - variation texture"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        variance = laplacian.var()
        return variance > threshold * 1000
    
    def check_triangle_shape(self, crop):
        """Pas triangle - analyse contours"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return False
        
        cnt = max(contours, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        
        # Triangle = 3 points
        return 2 <= len(approx) <= 4
    
    def check_alignment(self, crop, savon_mask, threshold=0.6):
        """Mal aligné - centrage dans box"""
        h, w = crop.shape[:2]
        moments = cv2.moments(savon_mask)
        
        if moments['m00'] == 0:
            return False
        
        cx = moments['m10'] / moments['m00']
        cy = moments['m01'] / moments['m00']
        center_x, center_y = w / 2, h / 2
        
        dist = np.sqrt((cx - center_x)**2 + (cy - center_y)**2)
        max_dist = (w + h) / 4 * threshold
        
        return dist < max_dist
    
    def check_packaging_present(self, crop, threshold=0.1):
        """Packaging absent - détecte pixels clairs"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        white_pixels = np.sum(gray > 200) / gray.size
        return white_pixels > threshold

print("✅ Classe AnomalyDetector OK!")

## CELL 10: MAIN PROCESSING VIDEO (⏱️ 5-10 min)

In [ ]:
print(f"📹 Ouverture vidéo: {VIDEO_PATH}")
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"   Résolution: {w}x{h}")
print(f"   FPS: {fps}")
print(f"   Frames: {total_frames}")

# Video writer
output_video = os.path.join(OUTPUT_DIR, "output_annotated.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

# Charger modèle
print(f"\n🤖 Chargement modèle...")
model = YOLO(MODEL_PATH)

tracker = SimpleTracker()
anomaly_detector = AnomalyDetector()

# Stats
counted_ids = set()
total_count = 0
frame_results = []
counting_line_pos = int(w * COUNTING_LINE_X)

print(f"\n⚙️  Processing vidéo en cours...\n")

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"   Frame {frame_idx}/{total_frames}")
    
    # DÉTECTION
    results = model(frame, conf=0.5, verbose=False)
    detections = []
    
    for det in results[0].boxes:
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        conf = float(det.conf[0])
        detections.append((x1, y1, x2, y2, conf))
    
    # TRACKING
    tracked = tracker.update(detections)
    
    # COMPTAGE + ANOMALIES
    frame_data = {
        'frame': frame_idx,
        'timestamp': frame_idx / fps,
        'detections': len(tracked),
        'anomalies': []
    }
    
    for x1, y1, x2, y2, track_id in tracked:
        # COMPTAGE
        cx = (x1 + x2) // 2
        if cx > counting_line_pos and track_id not in counted_ids:
            counted_ids.add(track_id)
            total_count += 1
            status = "✓"
        else:
            status = "→"
        
        # Extraction zone savon
        crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
        
        if crop.size > 0:
            # ANOMALIES
            anomalies_list = []
            
            if anomaly_detector.check_packaging_damaged(crop):
                anomalies_list.append("PACKAGING_DÉCHIRÉ")
            
            if not anomaly_detector.check_triangle_shape(crop):
                anomalies_list.append("PAS_TRIANGLE")
            
            gray_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, mask = cv2.threshold(gray_crop, 100, 255, cv2.THRESH_BINARY)
            if not anomaly_detector.check_alignment(crop, mask):
                anomalies_list.append("MAL_ALIGNÉ")
            
            if not anomaly_detector.check_packaging_present(crop):
                anomalies_list.append("PACKAGING_ABSENT")
            
            frame_data['anomalies'].append({
                'track_id': track_id,
                'anomalies': anomalies_list,
                'status': status
            })
            
            # DESSINER SUR FRAME
            color = (0, 255, 0) if not anomalies_list else (0, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"ID:{track_id} {status}"
            if anomalies_list:
                label += " ⚠️"
            cv2.putText(frame, label, (x1, y1 - 10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Ligne de comptage
    cv2.line(frame, (counting_line_pos, 0), (counting_line_pos, h), (0, 255, 255), 2)
    cv2.putText(frame, f"SORTIE | Comptage: {total_count}", 
               (counting_line_pos - 150, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    out.write(frame)
    frame_results.append(frame_data)

cap.release()
out.release()

print(f"\n✅ Processing terminé!")
print(f"✅ Total comptés: {total_count}")
print(f"✅ Vidéo annotée: {output_video}")

## CELL 11: Générer RAPPORT CSV

In [ ]:
# Analyser anomalies
anomaly_count = defaultdict(int)
savon_anomalies = defaultdict(list)

for frame_data in frame_results:
    for anom in frame_data['anomalies']:
        track_id = anom['track_id']
        for anom_type in anom['anomalies']:
            anomaly_count[anom_type] += 1
            savon_anomalies[track_id].append(anom_type)

# CSV résultats
report_data = {
    'Métrique': [
        'Total comptés',
        'Savons avec anomalies',
        'Packaging déchiré',
        'Pas triangle',
        'Mal aligné',
        'Packaging absent'
    ],
    'Valeur': [
        total_count,
        len(savon_anomalies),
        anomaly_count['PACKAGING_DÉCHIRÉ'],
        anomaly_count['PAS_TRIANGLE'],
        anomaly_count['MAL_ALIGNÉ'],
        anomaly_count['PACKAGING_ABSENT']
    ]
}

df = pd.DataFrame(report_data)
csv_path = os.path.join(OUTPUT_DIR, "report.csv")
df.to_csv(csv_path, index=False)

print(f"\n📊 RAPPORT FINAL:")
print("="*50)
print(df.to_string(index=False))
print("="*50)
print(f"\n💾 CSV sauvegardé: {csv_path}")

## CELL 12: Afficher résultats finaux

In [ ]:
print(f"\n🎉 RÉSULTATS FINALS")
print(f"="*60)
print(f"\n📊 Statistiques comptage:")
print(f"   • Total savons comptés: {total_count}")
print(f"   • Durée vidéo: {total_frames/fps:.1f} sec")
print(f"   • Savons/sec: {total_count/(total_frames/fps):.2f}")

print(f"\n⚠️  Anomalies détectées:")
print(f"   • Savons avec problèmes: {len(savon_anomalies)}")
print(f"   • Packaging déchiré: {anomaly_count['PACKAGING_DÉCHIRÉ']}")
print(f"   • Pas triangle: {anomaly_count['PAS_TRIANGLE']}")
print(f"   • Mal aligné: {anomaly_count['MAL_ALIGNÉ']}")
print(f"   • Packaging absent: {anomaly_count['PACKAGING_ABSENT']}")

print(f"\n📁 Fichiers générés:")
print(f"   ✓ {output_video}")
print(f"   ✓ {csv_path}")
print(f"\n" + "="*60)

---
## 📝 NOTES IMPORTANTES

### Si auto-annotation ne détecte pas bien:
**CELL 5** → Ajuster les plages HSV:
```python
lower_brown = np.array([10, 60, 50])      # Essayer d'autres valeurs
upper_brown = np.array([25, 255, 200])    # Par ex: [15, 100, 70] et [30, 200, 180]
```

### Si erreur mémoire GPU pendant training:
**CELL 7** → Réduire batch size:
```python
batch=4  # Au lieu de 8
```

### Ajuster position ligne comptage:
**CELL 3** → Modifier:
```python
COUNTING_LINE_X = 0.85  # 0.5 = milieu, 1.0 = très à droite
```

### Pour voir la vidéo:
Accéder à: `results/output_annotated.mp4`